# MRI Atlas Exploration with nibabel + nilearn

Load and visualize brain atlases: Schaefer, Yeo, Destrieux, MNI templates. Extract region time series from resting-state fMRI.

## Prerequisites
- Understanding of brain atlases, parcellations, and standard coordinate spaces (MNI)
- Familiarity with NIfTI neuroimaging file format
- Basic knowledge of resting-state fMRI and functional connectivity
- Python, NumPy, and matplotlib skills

## Setup
Run the cell below to install dependencies and download data.

In [ ]:
!pip install nibabel nilearn matplotlib numpy

In [ ]:
import nibabel as nib
import nilearn.plotting as nlp
import nilearn.datasets as nld
import matplotlib.pyplot as plt
import numpy as np

# Load and display MNI152 1mm brain using nilearn's built-in template
from nilearn.datasets import load_mni152_template
mni_img = load_mni152_template(resolution=1)
print(f'MNI152 shape: {mni_img.shape}, voxel size: {mni_img.header.get_zooms()}')
nlp.plot_anat(mni_img, title='MNI152 1mm standard brain', display_mode='ortho', cut_coords=(0,0,0))
plt.show()

In [ ]:
# Schaefer 200-parcel 7-network atlas
schaefer_dataset = nld.fetch_atlas_schaefer_2018(n_rois=200, yeo_networks=7, resolution_mm=2)
schaefer = nib.load(schaefer_dataset.maps)
unique_regions = np.unique(schaefer.get_fdata())
print(f'Schaefer 200: {len(unique_regions)-1} regions (0=background)')
nlp.plot_roi(schaefer, title='Schaefer 200 parcels / 7 networks', colorbar=True)
plt.show()

In [ ]:
# Yeo 7-network parcellation
yeo = nld.fetch_atlas_yeo_2011()
nlp.plot_roi(yeo.maps, title='Yeo 2011: 7 resting-state networks',
             cmap='Paired', colorbar=True)
plt.show()
print('Yeo 7 networks:', yeo.labels)

In [ ]:
# Extract region time-series from ADHD resting-state data using Schaefer atlas
from nilearn.maskers import NiftiLabelsMasker
adhd = nld.fetch_adhd(n_subjects=1)
masker = NiftiLabelsMasker(labels_img=schaefer, standardize=True, verbose=1)
time_series = masker.fit_transform(adhd.func[0])
print(f'Time series shape: {time_series.shape}  (timepoints \u00d7 regions)')

# Compute correlation matrix
from nilearn.connectome import ConnectivityMeasure
conn = ConnectivityMeasure(kind='correlation')
fc_matrix = conn.fit_transform([time_series])[0]
nlp.plot_matrix(fc_matrix, colorbar=True, vmax=0.8, vmin=-0.8,
                title='Functional connectivity \u2014 Schaefer 200 (ADHD200 RS-fMRI)')
plt.show()

## References

- Schaefer, A., Kong, R., Gordon, E. M., Laumann, T. O., Zuo, X. N., Holmes, A. J., ... & Yeo, B. T. T. (2018). Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity MRI. *Cerebral Cortex*, 28(9), 3095\u20133114. https://doi.org/10.1093/cercor/bhx179
- Yeo, B. T. T., Krienen, F. M., Sepulcre, J., Sabuncu, M. R., Lashkari, D., Hollinshead, M., ... & Buckner, R. L. (2011). The organization of the human cerebral cortex estimated by intrinsic functional connectivity. *Journal of Neurophysiology*, 106(3), 1125\u20131165.
- Fonov, V. S., Evans, A. C., Botteron, K., Almli, C. R., McKinstry, R. C., & Collins, D. L. (2011). Unbiased average age-appropriate atlases for pediatric studies. *NeuroImage*, 54(1), 313\u2013327.
- Abraham, A., Pedregosa, F., Eickenberg, M., Gervais, P., Mueller, A., Kossaifi, J., ... & Varoquaux, G. (2014). Machine learning for neuroimaging with scikit-learn. *Frontiers in Neuroinformatics*, 8, 14.